##0 Descarga de las librerias que se usaran


In [ ]:
import os
import kagglehub
import numpy as np
import pandas as pd

##1 División de datos y Escalado (Preparación para Deep Learning)

In [ ]:
# 1. Descarga de el  dataset de antes
path = kagglehub.dataset_download("fedesoriano/stroke-prediction-dataset")
files = os.listdir(path)
csv_files = [f for f in files if f.endswith('.csv')]
csv_path = os.path.join(path, csv_files[0])
df = pd.read_csv(csv_path)

# 2. Eliminación de id primero y luego imputación de bmi
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)

bmi_mean = df['bmi'].mean()
df['bmi'] = df['bmi'].fillna(bmi_mean)

# 3. Codificación One-Hot al final
df = pd.get_dummies(df)

print("Dataset preparado correctamente. Forma:", df.shape)

Using Colab cache for faster access to the 'stroke-prediction-dataset' dataset.
Dataset preparado correctamente. Forma: (5110, 22)


##2 Diseño de la Arquitectura y Selección de Funciones (Keras / TensorFlow)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Separar variable objetivo (y) y variables predictoras (X)
y = df['stroke']
X = df.drop(columns=['stroke'])

# 2. División train/test (80% y el 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=123
)

# 3. Escalado de las variables
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Tamaño X_train: {X_train_scaled.shape} | Tamaño X_test: {X_test_scaled.shape}")

Tamaño X_train: (4088, 21) | Tamaño X_test: (1022, 21)


## 3 Diseño de la Arquitectura y Selección de Funciones


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(units=30, activation='relu'),
    layers.Dense(units=15, activation='relu'),
    layers.Dense(units=1, activation='sigmoid')
])

print("┌" + "─"*58 + "┐")
print(f"│ {'CAPA (TIPO)':<22} │ {'OUTPUT SHAPE':<18} │ {'PARAM #':<10} │")
print("├" + "─"*58 + "┤")

total = 0
for l in model.layers:
    p = l.count_params()
    total += p
    print(f"│ {l.name + ' (' + l.__class__.__name__ + ')':<22} │ {str(l.output_shape):<18} │ {p:<10} │")

print("├" + "─"*58 + "┤")
print(f"│ Total de parámetros: {total:<37} │")
print("└" + "─"*58 + "┘")


┌──────────────────────────────────────────────────────────┐
│ CAPA (TIPO)            │ OUTPUT SHAPE       │ PARAM #    │
├──────────────────────────────────────────────────────────┤
│ dense_12 (Dense)       │ (None, 30)         │ 660        │
│ dense_13 (Dense)       │ (None, 15)         │ 465        │
│ dense_14 (Dense)       │ (None, 1)          │ 16         │
├──────────────────────────────────────────────────────────┤
│ Total de parámetros: 1141                                  │
└──────────────────────────────────────────────────────────┘


##4 Entrenamiento, Métricas y Optimizadores

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Compilación del modelo con orden de métricas reorganizado
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc')
    ]
)

# Callback con orden de argumentos invertido
early_stop = EarlyStopping(
    restore_best_weights=True,
    patience=10,
    monitor='val_loss'
)

# Entrenamiento del modelo (épocas ajustadas a 90)
history = model.fit(
    X_train_scaled,
    y_train,
    batch_size=32,
    epochs=90,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/90
103/103 ━━━━━━━━━━━━━━━━━━━━ 9s 36ms/step - accuracy: 0.9508 - auc: 0.8293 - loss: 0.1610 - recall: 0.0000e+00 - val_accuracy: 0.9535 - val_auc: 0.8118 - val_loss: 0.1573 - val_recall: 0.0000e+00
Epoch 2/90
103/103 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9508 - auc: 0.8435 - loss: 0.1571 - recall: 0.0000e+00 - val_accuracy: 0.9535 - val_auc: 0.8090 - val_loss: 0.1585 - val_recall: 0.0000e+00
Epoch 3/90
103/103 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9508 - auc: 0.8468 - loss: 0.1551 - recall: 0.0000e+00 - val_accuracy: 0.9535 - val_auc: 0.8074 - val_loss: 0.1594 - val_recall: 0.0000e+00
Epoch 4/90
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9508 - auc: 0.8546 - loss: 0.1532 - recall: 0.0000e+00 - val_accuracy: 0.9535 - val_auc: 0.8057 - val_loss: 0.1596 - val_recall: 0.0000e+00
Epoch 5/90
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9508 - auc: 0.8592 - loss: 0.1516 - recall: 0.0000e+00 - val_accuracy: 0.9535 - val_auc: 0.8058 - val_los